In [10]:
from pathlib import Path
script_path = Path.cwd() / 'train_model_v3_fix.py'
print(script_path)

c:\GitHubMain\GraduateWork\ml3\train_model_v3_fix.py


In [11]:
namespace = {}
exec(script_path.read_text(encoding='utf-8'), namespace)
build_feature_table = namespace['build_feature_table']
train_model = namespace['train_model']

In [12]:
datasets_root = Path.cwd() / 'datasets'
print('Datasets root:', datasets_root)
df, audit = build_feature_table(datasets_root)
print('✅ Rows:', len(df))
print('✅ Unique datasets:', df['dataset_id'].nunique())
print('🔍 Новые признаки в датасете:', [c for c in df.columns if 'relative' in c or 'ratio' in c or 'log' in c or 'n_children' in c])
df.head()

Datasets root: c:\GitHubMain\GraduateWork\ml3\datasets


✅ Rows: 1600
✅ Unique datasets: 39
🔍 Новые признаки в датасете: ['n_children', 'clade_size_log', 'relative_branch_length', 'depth_ratio']


,dataset_id,node_id,target_bootstrap,branch_length,depth,subtree_fraction,subtree_balance,mean_child_branch_length,std_child_branch_length,n_children,...,taxa_count,alignment_length,gap_fraction_global,gc_mean_global,gc_std_global,variable_site_fraction_global,gap_fraction_clade,gc_mean_clade,gc_std_clade,mean_pairwise_pdist_clade
0,dt001,dt001_node_0,99.2,0.025070,0.025070,12.0,0.916667,0.010455,0.010455,2.0,...,2.0,8.0,0.4375,0.5625,0.4375,1.0,NaN,NaN,NaN,NaN
1,dt001,dt001_node_1,99.9,0.177210,0.177210,1.0,0.000000,0.121400,0.121400,2.0,...,2.0,8.0,0.4375,0.5625,0.4375,1.0,NaN,NaN,NaN,NaN
2,dt001,dt001_node_2,99.2,0.020910,0.045980,11.5,0.913043,0.001212,0.001212,2.0,...,2.0,8.0,0.4375,0.5625,0.4375,1.0,NaN,NaN,NaN,NaN
3,dt001,dt001_node_3,45.6,0.002424,0.048403,11.0,0.909091,0.003286,0.002553,2.0,...,2.0,8.0,0.4375,0.5625,0.4375,1.0,NaN,NaN,NaN,NaN
4,dt001,dt001_node_4,74.0,0.005839,0.054242,10.5,0.904762,0.001904,0.001115,2.0,...,2.0,8.0,0.4375,0.5625,0.4375,1.0,NaN,NaN,NaN,NaN


In [13]:
import pandas as pd
pd.DataFrame(audit)

,dataset_id,alignment_file,tree_file,ok,error,n_rows
0,dt001,c:\GitHubMain\GraduateWork\ml3\datasets\dt001\...,c:\GitHubMain\GraduateWork\ml3\datasets\dt001\...,True,None,24.0
1,dt002,c:\GitHubMain\GraduateWork\ml3\datasets\dt002\...,c:\GitHubMain\GraduateWork\ml3\datasets\dt002\...,True,None,10.0
2,dt003,c:\GitHubMain\GraduateWork\ml3\datasets\dt003\...,c:\GitHubMain\GraduateWork\ml3\datasets\dt003\...,True,None,34.0
3,dt004,c:\GitHubMain\GraduateWork\ml3\datasets\dt004\...,c:\GitHubMain\GraduateWork\ml3\datasets\dt004\...,True,None,25.0
4,dt005,c:\GitHubMain\GraduateWork\ml3\datasets\dt005\...,c:\GitHubMain\GraduateWork\ml3\datasets\dt005\...,True,None,51.0
5,dt007,c:\GitHubMain\GraduateWork\ml3\datasets\dt007\...,c:\GitHubMain\GraduateWork\ml3\datasets\dt007\...,True,None,110.0
6,dt008,c:\GitHubMain\GraduateWork\ml3\datasets\dt008\...,c:\GitHubMain\GraduateWork\ml3\datasets\dt008\...,True,None,38.0
7,dt009,c:\GitHubMain\GraduateWork\ml3\datasets\dt009\...,c:\GitHubMain\GraduateWork\ml3\datasets\dt009\...,True,None,60.0
8,dt010,c:\GitHubMain\GraduateWork\ml3\datasets\dt010\...,c:\GitHubMain\GraduateWork\ml3\datasets\dt010\...,True,None,14.0
9,dt012,c:\GitHubMain\GraduateWork\ml3\datasets\dt012\...,c:\GitHubMain\GraduateWork\ml3\datasets\dt012\...,True,None,28.0


In [14]:
# train_model возвращает: model, metrics, feature_importance, pred_df
model, metrics, feature_importance, pred_df = train_model(df)

print("\n📊 CV Метрики:")
print(f"R²   : {metrics['cv_r2']:.4f}")
print(f"MAE  : {metrics['cv_mae']:.4f}")
print(f"RMSE : {metrics['cv_rmse']:.4f}")
print(f"Folds: {len(metrics['fold_metrics'])}")

print("\n🔝 Топ-5 признаков:")
display(feature_importance.head())

Fold 1: MAE=21.725, RMSE=29.713, R²=-0.011
Fold 2: MAE=17.061, RMSE=22.105, R²=0.609
Fold 3: MAE=15.730, RMSE=21.392, R²=0.580
Fold 4: MAE=12.317, RMSE=17.996, R²=0.632
Fold 5: MAE=15.442, RMSE=20.117, R²=0.648

📊 CV Метрики:
R²   : 0.4918
MAE  : 16.4551
RMSE : 22.2647
Folds: 5

🔝 Топ-5 признаков:


,feature,importance
8,relative_branch_length,0.334998
0,branch_length,0.206204
9,depth_ratio,0.087623
4,mean_child_branch_length,0.079777
7,clade_size_log,0.063143


In [15]:
import json
import joblib
from pathlib import Path

output_dir = Path.cwd() / "ml_outputs_v2_updated"
output_dir.mkdir(exist_ok=True)

# Сохраняем всё в новую папку, чтобы не смешивать со старыми артефактами
df.to_csv(output_dir / "node_dataset.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(audit).to_csv(output_dir / "dataset_audit.csv", index=False, encoding="utf-8-sig")
feature_importance.to_csv(output_dir / "feature_importance.csv", index=False, encoding="utf-8-sig")
pred_df.to_csv(output_dir / "test_predictions.csv", index=False, encoding="utf-8-sig")

joblib.dump(model, output_dir / "model_v2.pkl")

# metrics уже содержит feature_columns, просто сохраняем
with open(output_dir / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"✅ Всё сохранено в: {output_dir}")

✅ Всё сохранено в: c:\GitHubMain\GraduateWork\ml3\ml_outputs_v2_updated


In [16]:
print("Всего строк:", len(df))
print("Уникальных datasets:", df["dataset_id"].nunique())
print("Колонки:")
print(df.columns.tolist())

Всего строк: 1600
Уникальных datasets: 39
Колонки:
['dataset_id', 'node_id', 'target_bootstrap', 'branch_length', 'depth', 'subtree_fraction', 'subtree_balance', 'mean_child_branch_length', 'std_child_branch_length', 'n_children', 'clade_size_log', 'relative_branch_length', 'depth_ratio', 'taxa_count', 'alignment_length', 'gap_fraction_global', 'gc_mean_global', 'gc_std_global', 'variable_site_fraction_global', 'gap_fraction_clade', 'gc_mean_clade', 'gc_std_clade', 'mean_pairwise_pdist_clade']


In [17]:
df["target_bootstrap"].describe()

count    1600.000000
mean       61.175844
std        32.742062
min         0.000000
25%        33.247500
50%        63.000000
75%        97.000000
max       100.000000
Name: target_bootstrap, dtype: float64

In [18]:
metrics

{'n_rows_total': 1600,
 'n_datasets_total': 39,
 'cv_mae': 16.45511898220373,
 'cv_rmse': 22.264693578189082,
 'cv_r2': 0.4917853327362879,
 'feature_columns': ['branch_length',
  'depth',
  'subtree_fraction',
  'subtree_balance',
  'mean_child_branch_length',
  'std_child_branch_length',
  'n_children',
  'clade_size_log',
  'relative_branch_length',
  'depth_ratio',
  'taxa_count',
  'alignment_length',
  'gap_fraction_global',
  'gc_mean_global',
  'gc_std_global',
  'variable_site_fraction_global',
  'gap_fraction_clade',
  'gc_mean_clade',
  'gc_std_clade',
  'mean_pairwise_pdist_clade'],
 'fold_metrics': [{'fold': 1,
   'mae': 21.72546070236697,
   'rmse': np.float64(29.712781110845043),
   'r2': -0.011307366587187229},
  {'fold': 2,
   'mae': 17.060816304087115,
   'rmse': np.float64(22.10542719443125),
   'r2': 0.6093761856304255},
  {'fold': 3,
   'mae': 15.730121727690317,
   'rmse': np.float64(21.392362627359244),
   'r2': 0.5804055452225361},
  {'fold': 4,
   'mae': 12.316